In [ ]:
# ============================================================
# PATH CONFIGURATION — Adjust for your environment
# ============================================================
# These notebooks were originally developed on Google Colab.
# Paths referencing /content/drive/MyDrive/... are Colab-specific.
#
# For local execution, set ROOT_DIR to your project root:
#
#   import os
#   ROOT_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
#   # Then replace /content/drive/MyDrive/... paths with os.path.join(ROOT_DIR, ...)
#
# The authoritative dataset files are in: data/train.parquet, data/validation.parquet, data/test.parquet
# Images are in: data/images/
# ============================================================


In [ ]:
# ============================================================
# MODULE D
# Final Dataset Validation + Grouped Splits + Parquet Builder
# (Image + Text Modalities ONLY)
# ============================================================
#
# Project:
# Multi-Modal Prompt Injection Detection Agent
# for Insurance AI Systems
#
# Module D Responsibilities
# -------------------------
# 1. Read outputs generated by Modules A, B & C (FROZEN / READ-ONLY)
# 2. Build unified Image + Text malicious metadata
# 3. Validate the final dataset (duplicates, integrity, 1:1 mapping)
# 4. Detect broken images
# 5. Generate grouped train/validation/test splits using source_benign_id
# 6. Generate optimized parquet dataset
# 7. Generate statistics.json
# 8. Generate dataset_manifest.json
#
# NOTE
# ----
# Benchmark modalities: Image + Text ONLY.
# PDF generation has been intentionally removed because benign PDF
# mappings are unavailable after preprocessing.
# Modules A, B and C are FINAL and FROZEN (READ-ONLY).
# Module B finalized the strict 1:1 benign-to-malicious mapping.
# Module C has been finalized successfully.
# Runtime OCR remains unchanged.
# No OCR metadata is generated here.
#
# This module is fully resumable.
#
# ============================================================

print("="*80)
print("MODULE D : FINAL DATASET VALIDATION, SPLITS & PARQUET BUILDER")
print("="*80)


MODULE D : FINAL DATASET VALIDATION, SPLITS & PARQUET BUILDER


In [ ]:
# ============================================================
# Imports
# ============================================================

import os
import gc
import sys
import json
import time
import hashlib
import logging
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.options.mode.copy_on_write = True

gc.enable()

In [ ]:
# ============================================================
# Logger
# ============================================================

logger = logging.getLogger("MODULE_D")

logger.setLevel(logging.INFO)

if not logger.handlers:

    formatter = logging.Formatter(
        "[%(asctime)s] %(levelname)s : %(message)s",
        datefmt="%H:%M:%S"
    )

    stream_handler = logging.StreamHandler(sys.stdout)
    stream_handler.setFormatter(formatter)

    logger.addHandler(stream_handler)

logger.info("Logger initialized.")

[19:05:55] INFO : Logger initialized.


INFO:MODULE_D:Logger initialized.


In [ ]:
# ============================================================
# Reproducibility
# ============================================================

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

logger.info(f"Random Seed : {RANDOM_SEED}")

[19:05:55] INFO : Random Seed : 42


INFO:MODULE_D:Random Seed : 42


In [ ]:
# ============================================================
# Helper Utilities
# ============================================================

def file_exists(path) -> bool:
    """
    Safe file existence check.
    """
    return Path(path).exists()


def ensure_directory(path):
    """
    Create directory only if required.
    """
    Path(path).mkdir(
        parents=True,
        exist_ok=True
    )


def current_timestamp():
    """
    Returns current timestamp.
    """
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def md5_checksum(filepath):
    """
    Calculate MD5 checksum.

    Used only during validation.
    """

    hash_md5 = hashlib.md5()

    with open(filepath, "rb") as f:

        for chunk in iter(lambda: f.read(1024 * 1024), b""):

            hash_md5.update(chunk)

    return hash_md5.hexdigest()


def human_size(num):

    for unit in ["B", "KB", "MB", "GB", "TB"]:

        if num < 1024:

            return f"{num:.2f} {unit}"

        num /= 1024

    return f"{num:.2f} PB"

In [ ]:
# ============================================================
# Resume Helper
# ============================================================

def stage_completed(output_file):

    output_file = Path(output_file)

    return output_file.exists() and output_file.stat().st_size > 0


def print_stage(stage):

    print("\n" + "="*80)
    print(stage)
    print("="*80)

In [ ]:
# ============================================================
# Atomic CSV Writer
# Prevents corruption if runtime disconnects
# ============================================================

def atomic_csv_write(df, output_path):

    output_path = Path(output_path)

    temp_path = output_path.with_suffix(".tmp")

    df.to_csv(
        temp_path,
        index=False
    )

    os.replace(
        temp_path,
        output_path
    )

In [ ]:
# ============================================================
# Atomic Parquet Writer
# ============================================================

def atomic_parquet_write(df, output_path):

    output_path = Path(output_path)

    temp_path = output_path.with_suffix(".tmp.parquet")

    df.to_parquet(
        temp_path,
        index=False
    )

    os.replace(
        temp_path,
        output_path
    )

In [ ]:
# ============================================================
# Module Information
# ============================================================

logger.info("Module D initialized successfully.")
logger.info("Runtime OCR : ENABLED (Inference Only)")
logger.info("Resume Support : ENABLED")
logger.info("Duplicate Storage : DISABLED")
logger.info("Vectorized Processing : ENABLED")
logger.info("Atomic Writes : ENABLED")

print("\nModule D Environment Ready.\n")

[19:05:55] INFO : Module D initialized successfully.


INFO:MODULE_D:Module D initialized successfully.


[19:05:55] INFO : Runtime OCR : ENABLED (Inference Only)


INFO:MODULE_D:Runtime OCR : ENABLED (Inference Only)


[19:05:55] INFO : Resume Support : ENABLED


INFO:MODULE_D:Resume Support : ENABLED


... (3 outputs left out)

INFO:MODULE_D:Vectorized Processing : ENABLED


[19:05:55] INFO : Atomic Writes : ENABLED


INFO:MODULE_D:Atomic Writes : ENABLED



Module D Environment Ready.



In [ ]:
# ============================================================
# CONFIG Reconstruction
# ============================================================

print_stage("STEP 1 : CONFIG RECONSTRUCTION")

# ------------------------------------------------------------
# Root Directory
# ------------------------------------------------------------

try:
    from google.colab import drive

    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive", force_remount=False)

    ROOT = Path("/content/drive/MyDrive")

except Exception:
    ROOT = Path.cwd()

logger.info(f"Project Root : {ROOT}")


STEP 1 : CONFIG RECONSTRUCTION
Mounted at /content/drive
[19:06:18] INFO : Project Root : /content/drive/MyDrive


INFO:MODULE_D:Project Root : /content/drive/MyDrive


In [ ]:
# ============================================================
# Project Paths
# ============================================================

CONFIG = {

    # --------------------------------------------------------
    # Benign Dataset
    # --------------------------------------------------------

    "MASTER_BENIGN_DATASET":
        ROOT / "MASTER_BENIGN_DATASET",

    "TEXT_SOURCE":
        ROOT / "MASTER_BENIGN_DATASET"
             / "TEXT_SOURCES"
             / "benign_text.csv",

    "IMAGE_SOURCE":
        ROOT / "MASTER_BENIGN_DATASET"
             / "IMAGE_SOURCES",

    # --------------------------------------------------------
    # Payload Bank
    # --------------------------------------------------------

    "PAYLOAD_BANK":
    ROOT
    / "MASTER_MALICIOUS_DATASET"
    / "PAYLOAD_BANK"
    / "insurance_payload_bank.csv",

    # --------------------------------------------------------
    # Module B Output
    # --------------------------------------------------------

    "MALICIOUS_TEXT":
    ROOT
    / "MASTER_MALICIOUS_DATASET"
    / "TEXT"
    / "malicious_text.csv",

    # --------------------------------------------------------
    # Module C Output (Image Modality ONLY)
    # --------------------------------------------------------

    "MALICIOUS_IMAGES":
    ROOT
    / "MASTER_MALICIOUS_DATASET"
    / "IMAGES",

    "IMAGE_LABELS":
        ROOT
        / "MASTER_MALICIOUS_DATASET"
        / "LABELS"
        / "image_labels.csv",

    # --------------------------------------------------------
    # Module D Outputs
    # --------------------------------------------------------

    "PARQUET":
        ROOT
        / "MASTER_MALICIOUS_DATASET"
        / "PARQUET"
        / "malicious_dataset.parquet",

    "METADATA":
        ROOT
        / "MASTER_MALICIOUS_DATASET"
        / "metadata.csv",

    "MANIFEST":
        ROOT
        / "MASTER_MALICIOUS_DATASET"
        / "dataset_manifest.json",

    "STATISTICS":
        ROOT
        / "MASTER_MALICIOUS_DATASET"
        / "statistics.json",

    "VALIDATION_DIR":
        ROOT
        / "MASTER_MALICIOUS_DATASET"
        / "VALIDATION",

    "VALIDATION":
        ROOT
        / "MASTER_MALICIOUS_DATASET"
        / "VALIDATION"
        / "validation_report.csv",

    "SPLITS_DIR":
        ROOT
        / "MASTER_MALICIOUS_DATASET"
        / "SPLITS",

    "TRAIN_SPLIT":
        ROOT
        / "MASTER_MALICIOUS_DATASET"
        / "SPLITS"
        / "train.parquet",

    "VAL_SPLIT":
        ROOT
        / "MASTER_MALICIOUS_DATASET"
        / "SPLITS"
        / "validation.parquet",

    "TEST_SPLIT":
        ROOT
        / "MASTER_MALICIOUS_DATASET"
        / "SPLITS"
        / "test.parquet",

    "MASTER":
        ROOT
        / "MASTER_MALICIOUS_DATASET"

}


In [ ]:
# ============================================================
# Create Missing Output Folders
# (Modules A, B & C are FROZEN — their folders already exist
#  and must NOT be recreated. Only the folders required for
#  Module D's own outputs are created here.)
# ============================================================

OUTPUT_DIRS = [

    CONFIG["SPLITS_DIR"],

    CONFIG["PARQUET"].parent,

    CONFIG["VALIDATION_DIR"]

]

for directory in OUTPUT_DIRS:

    ensure_directory(directory)

logger.info("Output folders verified (SPLITS, PARQUET, VALIDATION).")


[19:29:32] INFO : Output folders verified (SPLITS, PARQUET, VALIDATION).


INFO:MODULE_D:Output folders verified (SPLITS, PARQUET, VALIDATION).


In [ ]:
# ============================================================
# Validate Required Inputs
# ============================================================

print_stage("STEP 2 : INPUT VALIDATION")

REQUIRED_INPUTS = {

    "Benign Text":
        CONFIG["TEXT_SOURCE"],

    "Payload Bank":
        CONFIG["PAYLOAD_BANK"],

    "Malicious Text":
        CONFIG["MALICIOUS_TEXT"]

}

missing_inputs = []

for name, path in REQUIRED_INPUTS.items():

    if path.exists():

        logger.info(f"[FOUND] {name}")

    else:

        logger.error(f"[MISSING] {name}")

        missing_inputs.append(name)

if missing_inputs:

    raise FileNotFoundError(

        "\nMissing Required Inputs:\n"

        + "\n".join(missing_inputs)

    )

logger.info("Input validation completed.")


STEP 2 : INPUT VALIDATION
[19:29:38] INFO : [FOUND] Benign Text


INFO:MODULE_D:[FOUND] Benign Text


[19:29:38] INFO : [FOUND] Payload Bank


INFO:MODULE_D:[FOUND] Payload Bank


[19:29:38] INFO : [FOUND] Malicious Text


INFO:MODULE_D:[FOUND] Malicious Text


[19:29:38] INFO : Input validation completed.


INFO:MODULE_D:Input validation completed.


In [ ]:
# ============================================================
# Resume Support
# ============================================================

print_stage("STEP 3 : RESUME STATUS")

STAGES = {

    "Metadata":
        CONFIG["METADATA"],

    "Parquet":
        CONFIG["PARQUET"],

    "Train Split":
        CONFIG["TRAIN_SPLIT"],

    "Validation Split":
        CONFIG["VAL_SPLIT"],

    "Test Split":
        CONFIG["TEST_SPLIT"],

    "Validation Report":
        CONFIG["VALIDATION"],

    "Statistics":
        CONFIG["STATISTICS"],

    "Manifest":
        CONFIG["MANIFEST"]

}

resume_status = {}

for stage, path in STAGES.items():

    completed = stage_completed(path)

    resume_status[stage] = completed

    if completed:

        logger.info(f"[SKIP] {stage} already completed.")

    else:

        logger.info(f"[RUN ] {stage} pending.")



STEP 3 : RESUME STATUS
[19:32:58] INFO : [RUN ] Metadata pending.


INFO:MODULE_D:[RUN ] Metadata pending.


[19:32:58] INFO : [RUN ] Parquet pending.


INFO:MODULE_D:[RUN ] Parquet pending.


[19:32:58] INFO : [RUN ] Train Split pending.


INFO:MODULE_D:[RUN ] Train Split pending.


... (6 outputs left out)

[19:32:58] INFO : [RUN ] Statistics pending.


INFO:MODULE_D:[RUN ] Statistics pending.


[19:32:58] INFO : [RUN ] Manifest pending.


INFO:MODULE_D:[RUN ] Manifest pending.


In [ ]:
# ============================================================
# Project Summary
# ============================================================

print("\n")

print("="*80)
print("MODULE D CONFIG SUMMARY")
print("="*80)

print(f"Project Root        : {ROOT}")
print(f"Master Dataset      : {CONFIG['MASTER']}")
print(f"Payload Bank        : {CONFIG['PAYLOAD_BANK'].name}")
print(f"Malicious Text      : {CONFIG['MALICIOUS_TEXT'].name}")
print(f"Image Directory     : {CONFIG['MALICIOUS_IMAGES'].name}")
print(f"Modalities          : Image + Text")

print("="*80)

logger.info("CONFIG reconstruction successful.")
logger.info("Resume support initialized.")
logger.info("Module D ready for metadata generation.")




MODULE D CONFIG SUMMARY
Project Root        : /content/drive/MyDrive
Master Dataset      : /content/drive/MyDrive/MASTER_MALICIOUS_DATASET
Payload Bank        : insurance_payload_bank.csv
Malicious Text      : malicious_text.csv
Image Directory     : IMAGES
Modalities          : Image + Text
[19:34:07] INFO : CONFIG reconstruction successful.


INFO:MODULE_D:CONFIG reconstruction successful.


[19:34:07] INFO : Resume support initialized.


INFO:MODULE_D:Resume support initialized.


[19:34:07] INFO : Module D ready for metadata generation.


INFO:MODULE_D:Module D ready for metadata generation.


In [ ]:
# ============================================================
# STEP 4 : LOAD MODULE B & MODULE C OUTPUTS
# ============================================================

print_stage("STEP 4 : LOADING GENERATED DATA")

# ------------------------------------------------------------
# Load Malicious Text Dataset
# ------------------------------------------------------------

logger.info("Loading malicious text dataset...")

malicious_df = pd.read_csv(
    CONFIG["MALICIOUS_TEXT"],
    low_memory=False
)

logger.info(f"Loaded {len(malicious_df):,} malicious text samples.")


# ------------------------------------------------------------
# Load Image Labels (Optional)
# ------------------------------------------------------------

image_labels = pd.DataFrame()

if CONFIG["IMAGE_LABELS"].exists():

    logger.info("Loading image labels...")

    image_labels = pd.read_csv(
        CONFIG["IMAGE_LABELS"],
        low_memory=False
    )

    logger.info(f"Loaded {len(image_labels):,} image labels.")

else:

    logger.warning("Image labels not found. Continuing...")



STEP 4 : LOADING GENERATED DATA
[19:35:35] INFO : Loading malicious text dataset...


INFO:MODULE_D:Loading malicious text dataset...


[19:35:40] INFO : Loaded 144,937 malicious text samples.


INFO:MODULE_D:Loaded 144,937 malicious text samples.


[19:35:40] INFO : Loading image labels...


INFO:MODULE_D:Loading image labels...


[19:35:41] INFO : Loaded 3,687 image labels.


INFO:MODULE_D:Loaded 3,687 image labels.


In [ ]:
# ============================================================
# REQUIRED COLUMNS VALIDATION
# ============================================================

print_stage("STEP 5 : VALIDATING SCHEMA")


REQUIRED_TEXT_COLUMNS = [

    "malicious_id",
    "source_benign_id",
    "payload_id",
    "attack_family",
    "attack_location",
    "visibility",
    "label"

]


missing_columns = [

    col for col in REQUIRED_TEXT_COLUMNS

    if col not in malicious_df.columns

]


if missing_columns:

    raise ValueError(

        "\nMissing Required Columns:\n"

        + "\n".join(missing_columns)

    )


logger.info("Malicious text schema validated.")


STEP 5 : VALIDATING SCHEMA
[19:37:40] INFO : Malicious text schema validated.


INFO:MODULE_D:Malicious text schema validated.


In [ ]:
# ============================================================
# REMOVE DUPLICATE RECORDS
# ============================================================

print_stage("STEP 6 : DUPLICATE VALIDATION")

before = len(malicious_df)

malicious_df.drop_duplicates(

    subset=["malicious_id"],

    inplace=True,

    ignore_index=True

)

removed = before - len(malicious_df)

logger.info(f"Duplicate malicious samples removed : {removed}")


STEP 6 : DUPLICATE VALIDATION
[19:40:12] INFO : Duplicate malicious samples removed : 0


INFO:MODULE_D:Duplicate malicious samples removed : 0


In [ ]:
# ============================================================
# MISSING VALUE VALIDATION
# ============================================================

critical_columns = [

    "malicious_id",
    "source_benign_id",
    "payload_id",
    "attack_family",
    "label"

]

missing_summary = (

    malicious_df[critical_columns]

    .isna()

    .sum()

    .sort_values(ascending=False)

)

print("\nMissing Values\n")
display(missing_summary.to_frame("Missing"))


Missing Values



,Missing
malicious_id,0
source_benign_id,0
payload_id,0
attack_family,0
label,0


In [ ]:
# ============================================================
# LABEL VALIDATION
# ============================================================

valid_labels = {1}

invalid_labels = malicious_df.loc[
    ~malicious_df["label"].isin(valid_labels)
]

if len(invalid_labels):

    logger.warning(
        f"Invalid labels found : {len(invalid_labels)}"
    )

else:

    logger.info("Labels validated.")

[19:43:54] INFO : Labels validated.


INFO:MODULE_D:Labels validated.


In [ ]:
# ============================================================
# IMAGE PATH VALIDATION
# ============================================================

if len(image_labels):

    if "output_path" in image_labels.columns:

        image_labels["exists"] = image_labels["output_path"].map(
            lambda x: Path(x).exists()
        )

        missing_images = (~image_labels["exists"]).sum()

        logger.info(
            f"Missing Images : {missing_images}"
        )

    else:

        logger.warning("output_path column not found in image labels.")

[19:48:20] INFO : Missing Images : 0


INFO:MODULE_D:Missing Images : 0


In [ ]:
# ============================================================
# DATASET SUMMARY
# ============================================================

print("\n")

print("="*80)
print("MODULE B/C OUTPUT SUMMARY")
print("="*80)

print(f"Malicious Text Samples : {len(malicious_df):,}")
print(f"Image Labels           : {len(image_labels):,}")

print("="*80)

logger.info("Module B & Module C outputs loaded successfully.")




MODULE B/C OUTPUT SUMMARY
Malicious Text Samples : 144,937
Image Labels           : 3,687
[19:48:27] INFO : Module B & Module C outputs loaded successfully.


INFO:MODULE_D:Module B & Module C outputs loaded successfully.


In [ ]:
# ============================================================
# STEP 7 : BUILD UNIFIED MULTIMODAL METADATA
# ============================================================

print_stage("STEP 7 : BUILDING UNIFIED METADATA")

# ------------------------------------------------------------
# Resume Support
# ------------------------------------------------------------

if stage_completed(CONFIG["METADATA"]):

    logger.info("Unified metadata already exists.")

    unified_metadata = pd.read_csv(
        CONFIG["METADATA"],
        low_memory=False
    )

    logger.info(
        f"Loaded {len(unified_metadata):,} metadata records."
    )

else:

    logger.info("Creating unified metadata...")


STEP 7 : BUILDING UNIFIED METADATA
[19:49:42] INFO : Creating unified metadata...


INFO:MODULE_D:Creating unified metadata...


In [ ]:
if not image_labels.empty:

    image_df = image_labels.copy()

    rename_map = {}

    if "id" in image_df.columns:
        rename_map["id"] = "malicious_id"

    if "image_file" in image_df.columns:
        rename_map["image_file"] = "image_path"

    if "filename" in image_df.columns:
        rename_map["filename"] = "image_path"

    # <-- Ye line add karni hai
    if "output_path" in image_df.columns:
        rename_map["output_path"] = "image_path"

    image_df.rename(
        columns=rename_map,
        inplace=True
    )

else:

    image_df = pd.DataFrame(
        columns=[
            "malicious_id",
            "image_path"
        ]
    )

In [ ]:
# ============================================================
# Merge All Modalities (Image + Text)
# ============================================================

unified_metadata = (

    malicious_df

    .merge(

        image_df,

        how="left",

        on="malicious_id"

    )

)


In [ ]:
# ============================================================
# Additional Metadata
# ============================================================

unified_metadata["generation_timestamp"] = current_timestamp()

unified_metadata["runtime_ocr"] = True

unified_metadata["dataset_version"] = "V2"

unified_metadata["dataset_type"] = "MASTER_MALICIOUS_DATASET"

unified_metadata["module"] = "MODULE_D"

In [ ]:
# ============================================================
# Remove Duplicate Records
# ============================================================

before = len(unified_metadata)

unified_metadata.drop_duplicates(

    subset=["malicious_id"],

    inplace=True,

    ignore_index=True

)

after = len(unified_metadata)

logger.info(
    f"Duplicate metadata removed : {before-after}"
)

[20:02:54] INFO : Duplicate metadata removed : 0


INFO:MODULE_D:Duplicate metadata removed : 0


In [ ]:
# ============================================================
# Sort Dataset
# ============================================================

sort_columns = [
    col for col in [
        "attack_family",
        "payload_id",
        "malicious_id"
    ]
    if col in unified_metadata.columns
]

if sort_columns:

    unified_metadata.sort_values(
        by=sort_columns,
        inplace=True,
        ignore_index=True
    )

In [ ]:
# ============================================================
# Save Metadata
# ============================================================

atomic_csv_write(

    unified_metadata,

    CONFIG["METADATA"]

)

logger.info(
    "Unified metadata saved successfully."
)

[20:05:35] INFO : Unified metadata saved successfully.


INFO:MODULE_D:Unified metadata saved successfully.


In [ ]:
# ============================================================
# Standardize Image Labels
# ============================================================

if not image_labels.empty:

    image_df = image_labels.copy()

    rename_map = {}

    if "id" in image_df.columns:
        rename_map["id"] = "malicious_id"

    if "image_file" in image_df.columns:
        rename_map["image_file"] = "image_path"

    if "filename" in image_df.columns:
        rename_map["filename"] = "image_path"

    if "output_path" in image_df.columns:
        rename_map["output_path"] = "image_path"

    image_df.rename(
        columns=rename_map,
        inplace=True
    )

    # --------------------------------------------------------
    # Keep ONLY image-specific metadata
    # --------------------------------------------------------

    keep_cols = [
        "malicious_id",
        "image_path",
        "image_width",
        "image_height",
        "file_checksum",
        "status",
        "error",
        "image_generation_timestamp",
        "exists"
    ]

    keep_cols = [c for c in keep_cols if c in image_df.columns]

    image_df = image_df[keep_cols].copy()

else:

    image_df = pd.DataFrame(
        columns=[
            "malicious_id",
            "image_path",
            "image_width",
            "image_height",
            "file_checksum",
            "status",
            "error",
            "image_generation_timestamp",
            "exists"
        ]
    )


# ============================================================
# Merge All Modalities (Image + Text)
# ============================================================

unified_metadata = malicious_df.merge(
    image_df,
    how="left",
    on="malicious_id",
    validate="one_to_one"
)

In [ ]:
if not unified_metadata["malicious_id"].is_unique:
    raise ValueError("Duplicate malicious_id found after multimodal merge.")

In [ ]:
# ============================================================
# STEP 8 : BUILD OPTIMIZED PARQUET DATASET
# ============================================================

print_stage("STEP 8 : BUILDING OPTIMIZED PARQUET DATASET")

# ------------------------------------------------------------
# Resume Support
# ------------------------------------------------------------

if stage_completed(CONFIG["PARQUET"]):

    logger.info("Optimized Parquet already exists. Loading existing file...")

    unified_metadata = pd.read_parquet(CONFIG["PARQUET"])

    logger.info(f"Loaded {len(unified_metadata):,} records.")

else:

    logger.info("Preparing optimized dataset...")

    # --------------------------------------------------------
    # Memory Before Optimization
    # --------------------------------------------------------

    memory_before = unified_metadata.memory_usage(deep=True).sum()

    logger.info(
        f"Memory Before Optimization : {human_size(memory_before)}"
    )

    # --------------------------------------------------------
    # Remove Duplicate Records
    # --------------------------------------------------------

    before = len(unified_metadata)

    unified_metadata.drop_duplicates(
        subset=["malicious_id"],
        keep="first",
        inplace=True,
        ignore_index=True
    )

    duplicates_removed = before - len(unified_metadata)

    logger.info(
        f"Duplicate Records Removed : {duplicates_removed}"
    )

    # --------------------------------------------------------
    # Remove Empty Rows
    # --------------------------------------------------------

    unified_metadata.dropna(
        subset=["malicious_id", "payload_id"],
        inplace=True
    )

    unified_metadata.reset_index(
        drop=True,
        inplace=True
    )

    # --------------------------------------------------------
    # Optimize String Columns
    # --------------------------------------------------------

    string_columns = [
        "malicious_id",
        "source_benign_id",
        "payload_id",
        "attack_name",
        "attack_family",
        "attack_location",
        "payload_text",
        "visibility",
        "severity",
        "image_path"
    ]

    for col in string_columns:

        if col in unified_metadata.columns:

            unified_metadata[col] = (
                unified_metadata[col]
                .fillna("")
                .astype("string")
            )

    # --------------------------------------------------------
    # Optimize Category Columns
    # --------------------------------------------------------

    category_columns = [
        "attack_family",
        "attack_location",
        "visibility",
        "severity",
        "dataset_type",
        "dataset_version",
        "module"
    ]

    for col in category_columns:

        if col in unified_metadata.columns:

            unified_metadata[col] = (
                unified_metadata[col]
                .astype("category")
            )

    # --------------------------------------------------------
    # Optimize Boolean Columns
    # --------------------------------------------------------

    boolean_columns = [
    "runtime_ocr",
    "exists"
]

    for col in boolean_columns:

        if col in unified_metadata.columns:

            unified_metadata[col] = (
                unified_metadata[col]
                .astype(bool)
            )

    # --------------------------------------------------------
    # Sort Dataset
    # --------------------------------------------------------

    sort_columns = [
        c for c in [
            "attack_family",
            "payload_id",
            "malicious_id"
        ]
        if c in unified_metadata.columns
    ]

    if sort_columns:

        unified_metadata.sort_values(
            by=sort_columns,
            inplace=True,
            ignore_index=True
        )

    # --------------------------------------------------------
    # Required Column Validation
    # --------------------------------------------------------

    required_columns = [
        "malicious_id",
        "source_benign_id",
        "payload_id",
        "attack_family",
        "label"
    ]

    missing_columns = [
        c for c in required_columns
        if c not in unified_metadata.columns
    ]

    if missing_columns:

        raise ValueError(
            "Missing Required Columns:\n"
            + "\n".join(missing_columns)
        )

    # --------------------------------------------------------
    # Missing Value Validation
    # --------------------------------------------------------

    missing_summary = (
        unified_metadata[required_columns]
        .isna()
        .sum()
    )

    if missing_summary.sum():

        logger.warning("Missing values detected:")

        print(missing_summary)

    else:

        logger.info("No missing values detected.")

    # --------------------------------------------------------
    # Memory After Optimization
    # --------------------------------------------------------

    memory_after = unified_metadata.memory_usage(
        deep=True
    ).sum()

    reduction = (
        (memory_before - memory_after)
        / memory_before
    ) * 100

    logger.info(
        f"Memory After Optimization : {human_size(memory_after)}"
    )

    logger.info(
        f"Memory Reduction : {reduction:.2f}%"
    )

    # --------------------------------------------------------
    # Save Optimized Parquet
    # --------------------------------------------------------

    unified_metadata.to_parquet(
        CONFIG["PARQUET"],
        index=False,
        compression="snappy"
    )

    logger.info("Optimized Parquet Dataset Saved Successfully.")

# ============================================================
# Dataset Statistics
# ============================================================

DATASET_STATS = {

    "Total Samples":
        len(unified_metadata),

    "Unique Payloads":
        unified_metadata["payload_id"].nunique(),

    "Attack Families":
        unified_metadata["attack_family"].nunique(),

    "Labels":
        unified_metadata["label"].nunique(),

    "Image Samples":
        (
            unified_metadata["image_path"].ne("").sum()
            if "image_path" in unified_metadata.columns
            else 0
        ),

    "Dataset Size":
        human_size(CONFIG["PARQUET"].stat().st_size)

}

print("\n" + "=" * 80)
print("OPTIMIZED PARQUET DATASET SUMMARY")
print("=" * 80)

for key, value in DATASET_STATS.items():
    print(f"{key:<25}: {value}")

print("=" * 80)

logger.info("STEP 8 COMPLETED SUCCESSFULLY.")



STEP 8 : BUILDING OPTIMIZED PARQUET DATASET
[20:18:18] INFO : Preparing optimized dataset...


INFO:MODULE_D:Preparing optimized dataset...


[20:18:18] INFO : Memory Before Optimization : 279.91 MB


INFO:MODULE_D:Memory Before Optimization : 279.91 MB


[20:18:18] INFO : Duplicate Records Removed : 0


INFO:MODULE_D:Duplicate Records Removed : 0


... (6 outputs left out)

[20:18:21] INFO : Optimized Parquet Dataset Saved Successfully.


INFO:MODULE_D:Optimized Parquet Dataset Saved Successfully.



OPTIMIZED PARQUET DATASET SUMMARY
Total Samples            : 144937
Unique Payloads          : 58
Attack Families          : 15
Labels                   : 1
Image Samples            : 3687
Dataset Size             : 23.75 MB
[20:18:21] INFO : STEP 8 COMPLETED SUCCESSFULLY.


INFO:MODULE_D:STEP 8 COMPLETED SUCCESSFULLY.


In [ ]:
# ============================================================
# STEP 9 : GENERATE GROUPED TRAIN / VALIDATION / TEST SPLITS
# Splits are grouped by source_benign_id so that every sample
# derived from the same benign document stays within a single
# split (prevents data leakage across train/val/test).
# ============================================================

print_stage("STEP 9 : GENERATING GROUPED DATA SPLITS")

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

SPLIT_FILES = [
    CONFIG["TRAIN_SPLIT"],
    CONFIG["VAL_SPLIT"],
    CONFIG["TEST_SPLIT"]
]

if all(stage_completed(p) for p in SPLIT_FILES):

    logger.info("Grouped splits already exist. Loading existing files...")

    train_df = pd.read_parquet(CONFIG["TRAIN_SPLIT"])
    val_df = pd.read_parquet(CONFIG["VAL_SPLIT"])
    test_df = pd.read_parquet(CONFIG["TEST_SPLIT"])

else:

    logger.info("Generating grouped splits using source_benign_id...")

    unique_groups = (
    unified_metadata["source_benign_id"]
    .dropna()
    .astype(str)
    .unique()
)

    unique_groups = np.array(sorted(unique_groups.tolist()))

    rng = np.random.RandomState(RANDOM_SEED)

    shuffled_groups = unique_groups.copy()

    rng.shuffle(shuffled_groups)

    n_groups = len(shuffled_groups)

    train_end = int(n_groups * TRAIN_RATIO)
    val_end = train_end + int(n_groups * VAL_RATIO)

    train_groups = set(shuffled_groups[:train_end])
    val_groups = set(shuffled_groups[train_end:val_end])
    test_groups = set(shuffled_groups[val_end:])

    def assign_split(group_id):

        if group_id in train_groups:
            return "train"

        if group_id in val_groups:
            return "validation"

        if group_id in test_groups:
            return "test"

        raise ValueError(
    f"Unknown benign group encountered: {group_id}"
)

    unified_metadata["split"] = (
        unified_metadata["source_benign_id"]
        .map(assign_split)
    )

    train_df = (
        unified_metadata[unified_metadata["split"] == "train"]
        .reset_index(drop=True)
    )

    val_df = (
        unified_metadata[unified_metadata["split"] == "validation"]
        .reset_index(drop=True)
    )

    test_df = (
        unified_metadata[unified_metadata["split"] == "test"]
        .reset_index(drop=True)
    )

    atomic_parquet_write(train_df, CONFIG["TRAIN_SPLIT"])
    atomic_parquet_write(val_df, CONFIG["VAL_SPLIT"])
    atomic_parquet_write(test_df, CONFIG["TEST_SPLIT"])

    logger.info("Grouped splits generated and saved successfully.")

# ------------------------------------------------------------
# Group Leakage Verification
# ------------------------------------------------------------

train_ids = set(train_df["source_benign_id"].unique())
val_ids = set(val_df["source_benign_id"].unique())
test_ids = set(test_df["source_benign_id"].unique())

leakage = (
    (train_ids & val_ids)
    | (train_ids & test_ids)
    | (val_ids & test_ids)
)

if leakage:

    logger.error(
        f"Group leakage detected across splits : {len(leakage)} ids"
    )

else:

    logger.info("No group leakage detected across splits.")

# ------------------------------------------------------------
# Split Summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DATA SPLIT SUMMARY")
print("=" * 80)

print(f"Train Samples            : {len(train_df):,}")
print(f"Validation Samples       : {len(val_df):,}")
print(f"Test Samples             : {len(test_df):,}")
print(f"Train Benign Groups      : {len(train_ids):,}")
print(f"Validation Benign Groups : {len(val_ids):,}")
print(f"Test Benign Groups       : {len(test_ids):,}")

print("=" * 80)

logger.info("STEP 9 COMPLETED SUCCESSFULLY.")



STEP 9 : GENERATING GROUPED DATA SPLITS
[20:23:27] INFO : Generating grouped splits using source_benign_id...


INFO:MODULE_D:Generating grouped splits using source_benign_id...


[20:23:29] INFO : Grouped splits generated and saved successfully.


INFO:MODULE_D:Grouped splits generated and saved successfully.


[20:23:29] INFO : No group leakage detected across splits.


INFO:MODULE_D:No group leakage detected across splits.



DATA SPLIT SUMMARY
Train Samples            : 115,949
Validation Samples       : 14,493
Test Samples             : 14,495
Train Benign Groups      : 115,949
Validation Benign Groups : 14,493
Test Benign Groups       : 14,495
[20:23:29] INFO : STEP 9 COMPLETED SUCCESSFULLY.


INFO:MODULE_D:STEP 9 COMPLETED SUCCESSFULLY.


In [ ]:
# ============================================================
# STEP 10 : GENERATE STATISTICS.JSON
# ============================================================

print_stage("STEP 10 : GENERATING DATASET STATISTICS")

if stage_completed(CONFIG["STATISTICS"]):

    logger.info("statistics.json already exists. Skipping regeneration.")

else:

    logger.info("Computing dataset statistics...")

    statistics_report = {

        "generated_on":
            current_timestamp(),

        "modalities":
            ["image", "text"],

        "total_samples":
            int(len(unified_metadata)),

        "unique_benign_documents":
            int(unified_metadata["source_benign_id"].nunique()),

        "unique_payloads":
            int(unified_metadata["payload_id"].nunique()),

        "attack_families":
            int(unified_metadata["attack_family"].nunique()),
        "total_attack_types":
             int(unified_metadata["attack_name"].nunique()),


        "label_distribution":
            unified_metadata["label"]
            .value_counts()
            .to_dict(),

        "attack_family_distribution":
            unified_metadata["attack_family"]
            .value_counts()
            .to_dict(),

        "image_samples":
            int(
                unified_metadata["image_path"]
                .fillna("")
                .ne("")
                .sum()
            )
            if "image_path" in unified_metadata.columns
            else 0,

        "image_coverage_percent":
            round(
                (
                    unified_metadata["image_path"]
                    .fillna("")
                    .ne("")
                    .mean()
                ) * 100,
                2
            )
            if "image_path" in unified_metadata.columns
            else 0,




        "splits": {

            "train": int(len(train_df)),

            "validation": int(len(val_df)),

            "test": int(len(test_df))

        }

    }
    with open(
        CONFIG["STATISTICS"],
        "w",
        encoding="utf-8"
    ) as fp:

        json.dump(
            statistics_report,
            fp,
            indent=4,
            ensure_ascii=False
        )

    logger.info("statistics.json saved successfully.")

print("\n" + "=" * 80)
print("STATISTICS.JSON GENERATED")
print("=" * 80)
print(f"Path : {CONFIG['STATISTICS']}")
print("=" * 80)

logger.info("STEP 10 COMPLETED SUCCESSFULLY.")



STEP 10 : GENERATING DATASET STATISTICS
[20:39:10] INFO : Computing dataset statistics...


INFO:MODULE_D:Computing dataset statistics...


[20:39:11] INFO : statistics.json saved successfully.


INFO:MODULE_D:statistics.json saved successfully.



STATISTICS.JSON GENERATED
Path : /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/statistics.json
[20:39:11] INFO : STEP 10 COMPLETED SUCCESSFULLY.


INFO:MODULE_D:STEP 10 COMPLETED SUCCESSFULLY.


In [ ]:
# ============================================================
# STEP 11 : GENERATE DATASET MANIFEST
# (MASTER_MALICIOUS_DATASET itself is NOT recreated here —
#  it was already built by Modules A, B & C, which are FROZEN.
#  This step only validates required outputs and writes the
#  dataset manifest.)
# ============================================================

print_stage("STEP 11 : GENERATING DATASET MANIFEST")

master_dir = CONFIG["MASTER"]

# ------------------------------------------------------------
# Files that must exist
# ------------------------------------------------------------

required_outputs = {

    "Malicious Text CSV" : CONFIG["MALICIOUS_TEXT"],
    "Metadata CSV"       : CONFIG["METADATA"],
    "Parquet Dataset"    : CONFIG["PARQUET"],
    "Train Split"        : CONFIG["TRAIN_SPLIT"],
    "Validation Split"   : CONFIG["VAL_SPLIT"],
    "Test Split"         : CONFIG["TEST_SPLIT"],
    "Statistics"         : CONFIG["STATISTICS"]

}

optional_outputs = {

    "Image Labels" : CONFIG["IMAGE_LABELS"]

}

# ------------------------------------------------------------
# Validate Required Outputs
# ------------------------------------------------------------

logger.info("Validating required outputs...")

missing_required = []

for name, path in required_outputs.items():

    if Path(path).exists():

        logger.info(f"[FOUND] {name}")

    else:

        logger.error(f"[MISSING] {name}")

        missing_required.append(name)

if missing_required:

    raise FileNotFoundError(

        "\nRequired outputs missing:\n"

        + "\n".join(missing_required)

    )

# ------------------------------------------------------------
# Validate Optional Outputs
# ------------------------------------------------------------

logger.info("Checking optional outputs...")

for name, path in optional_outputs.items():

    if Path(path).exists():

        logger.info(f"[FOUND] {name}")

    else:

        logger.warning(f"[OPTIONAL] {name} not found.")

# ------------------------------------------------------------
# Dataset Manifest
# ------------------------------------------------------------

logger.info("Generating dataset manifest...")

manifest = {

    "project" : "Multi-Modal Prompt Injection Detection Agent",

    "dataset_name" : "MASTER_MALICIOUS_DATASET",

    "dataset_version" : "V2",

    "modalities" : ["image", "text"],

    "generated_on" : current_timestamp(),

    "runtime_ocr" : True,

    "compression" : "snappy",

    "storage_policy" : "Single Copy",

    "resume_supported" : True,

    "total_samples" : int(len(unified_metadata)),

    "unique_payloads" : int(

        unified_metadata["payload_id"].nunique()

    ),

    "attack_families" : int(

        unified_metadata["attack_family"].nunique()

    ),

    "attack_types" : int(

       unified_metadata["attack_name"].nunique()

   ),

    "labels" : (

        unified_metadata["label"]

        .value_counts()

        .to_dict()

    ),

        "image_samples" : int(

        unified_metadata["image_path"]

        .fillna("")

        .ne("")

        .sum()

    )

    if "image_path" in unified_metadata.columns

    else 0,

    "parquet_compression" : "snappy",

    "splits" : {

        "train" : int(len(train_df)),

        "validation" : int(len(val_df)),

        "test" : int(len(test_df))

    }

}

with open(

    CONFIG["MANIFEST"],

    "w",

    encoding="utf-8"

) as fp:

    json.dump(

        manifest,

        fp,

        indent=4,

        ensure_ascii=False

    )

logger.info("dataset_manifest.json saved.")

# ------------------------------------------------------------
# Directory Statistics
# ------------------------------------------------------------

logger.info("Computing directory statistics...")

image_count = 0

if CONFIG["MALICIOUS_IMAGES"].exists():

    image_count = len(

        list(

            CONFIG["MALICIOUS_IMAGES"].glob("*")

        )

    )

# ------------------------------------------------------------
# Storage Statistics
# ------------------------------------------------------------

dataset_size = 0

for file in master_dir.rglob("*"):

    if file.is_file():

        dataset_size += file.stat().st_size

# ------------------------------------------------------------
# Final Summary
# ------------------------------------------------------------

print()

print("="*80)
print("DATASET MANIFEST SUMMARY")
print("="*80)

print(f"Dataset Directory     : {master_dir}")

print(f"Metadata Records      : {len(unified_metadata):,}")

print(f"Image Files           : {image_count:,}")

print(f"Attack Families       : {unified_metadata['attack_family'].nunique()}")

print(

    f"Attack Types          : {unified_metadata['attack_name'].nunique()}"

)

print(f"Unique Payloads       : {unified_metadata['payload_id'].nunique()}")

print(f"Dataset Size          : {human_size(dataset_size)}")

print(f"Manifest              : {CONFIG['MANIFEST'].name}")

print("="*80)

logger.info("dataset_manifest.json generated successfully.")



STEP 11 : GENERATING DATASET MANIFEST
[20:49:02] INFO : Validating required outputs...


INFO:MODULE_D:Validating required outputs...


[20:49:02] INFO : [FOUND] Malicious Text CSV


INFO:MODULE_D:[FOUND] Malicious Text CSV


[20:49:02] INFO : [FOUND] Metadata CSV


INFO:MODULE_D:[FOUND] Metadata CSV


... (18 outputs left out)

[20:49:02] INFO : Computing directory statistics...


INFO:MODULE_D:Computing directory statistics...



DATASET MANIFEST SUMMARY
Dataset Directory     : /content/drive/MyDrive/MASTER_MALICIOUS_DATASET
Metadata Records      : 144,937
Image Files           : 3,687
Attack Families       : 15
Attack Types          : 58
Unique Payloads       : 58
Dataset Size          : 1.72 GB
Manifest              : dataset_manifest.json
[20:49:04] INFO : dataset_manifest.json generated successfully.


INFO:MODULE_D:dataset_manifest.json generated successfully.


In [ ]:
# ============================================================
# STEP 12 : FINAL DATASET VALIDATION
# ============================================================

print_stage("STEP 12 : FINAL DATASET VALIDATION")

validation_results = []

logger.info("Running complete dataset validation...")

# ============================================================
# Helper Function
# ============================================================

def add_result(check_name, passed, count, remarks=""):

    validation_results.append({

        "check": check_name,

        "status": "PASS" if passed else "FAIL",

        "count": int(count),

        "remarks": remarks

    })

# ============================================================
# 1. Duplicate malicious_id
# ============================================================

duplicate_ids = unified_metadata.duplicated(
    subset=["malicious_id"]
).sum()

add_result(
    "Duplicate malicious_id",
    duplicate_ids == 0,
    duplicate_ids
)

# ============================================================
# 2. Missing Metadata
# ============================================================

required_metadata = [

    "malicious_id",
    "source_benign_id",
    "payload_id",
    "attack_family",
    "label"

]

missing_metadata = int(

    unified_metadata[required_metadata]

    .isna()

    .sum()

    .sum()

)

add_result(

    "Missing Metadata",

    missing_metadata == 0,

    missing_metadata

)

# ============================================================
# 3. Missing Labels
# ============================================================

missing_labels = int(

    unified_metadata["label"]

    .isna()

    .sum()

)

add_result(

    "Missing Labels",

    missing_labels == 0,

    missing_labels

)

# ============================================================
# 4. Invalid Labels
# ============================================================

valid_labels = {1}

invalid_labels = int(

    (~unified_metadata["label"]

    .isin(valid_labels)).sum()

)

add_result(

    "Invalid Labels",

    invalid_labels == 0,

    invalid_labels

)

# ============================================================
# 5. Empty Payloads
# ============================================================

if "payload_text" in unified_metadata.columns:

    empty_payloads = int(

        unified_metadata["payload_text"]

        .fillna("")

        .str.strip()

        .eq("")

        .sum()

    )

else:

    empty_payloads = 0

add_result(

    "Empty Payload Text",

    empty_payloads == 0,

    empty_payloads

)

# ============================================================
# 6. Invalid Attack Families
# ============================================================

valid_families = set(

    pd.read_csv(CONFIG["PAYLOAD_BANK"])["attack_family"]

    .unique()

)

invalid_families = int(

    (~unified_metadata["attack_family"]

    .isin(valid_families)).sum()

)

add_result(

    "Invalid Attack Family",

    invalid_families == 0,

    invalid_families

)

# ============================================================
# 7. Broken Image Paths
# ============================================================

broken_images = 0

if "image_path" in unified_metadata.columns:

    image_paths = unified_metadata["image_path"].fillna("")

    broken_images = int(

        (
            image_paths.ne("")

            &

            (~image_paths.map(lambda x: Path(x).exists()))

        ).sum()

    )

add_result(

    "Broken Image Paths",

    broken_images == 0,

    broken_images

)

# ============================================================
# 8. One-to-One Mapping
# ============================================================

duplicate_source = int(

    unified_metadata

    .duplicated(

        subset=[

            "malicious_id"

        ]

    )

    .sum()

)

add_result(

    "One-to-One Mapping",

    duplicate_source == 0,

    duplicate_source

)

# ============================================================
# 9. Orphan Images
# ============================================================

orphan_images = 0

if CONFIG["MALICIOUS_IMAGES"].exists():

    metadata_images = set(

        unified_metadata["image_path"]

        .dropna()

        .astype(str)

        .apply(lambda x: Path(x).name)

    )

    disk_images = set(

        f.name

        for f in CONFIG["MALICIOUS_IMAGES"].glob("*")

        if f.is_file()

    )

    orphan_images = len(

        disk_images - metadata_images

    )

add_result(

    "Orphan Images",

    orphan_images == 0,

    orphan_images

)

# ============================================================
# 10. Group Leakage Across Splits
# ============================================================

split_leakage = len(
    (train_ids & val_ids)
    | (train_ids & test_ids)
    | (val_ids & test_ids)
)

add_result(

    "Split Group Leakage",

    split_leakage == 0,

    split_leakage

)

# ============================================================
# Validation Report
# ============================================================

validation_df = pd.DataFrame(validation_results)

atomic_csv_write(

    validation_df,

    CONFIG["VALIDATION"]

)

# ============================================================
# Summary
# ============================================================

passed = (validation_df["status"] == "PASS").sum()

failed = (validation_df["status"] == "FAIL").sum()

print("\n")

print("="*90)

print("FINAL DATASET VALIDATION SUMMARY")

print("="*90)

display(validation_df)

print("="*90)

print(f"Total Checks : {len(validation_df)}")

print(f"Passed       : {passed}")

print(f"Failed       : {failed}")

print("="*90)

if failed == 0:

    logger.info("Dataset validation PASSED.")

else:

    logger.warning("Dataset validation completed with warnings.")

logger.info("Validation report saved successfully.")



STEP 12 : FINAL DATASET VALIDATION
[20:55:16] INFO : Running complete dataset validation...


INFO:MODULE_D:Running complete dataset validation...




FINAL DATASET VALIDATION SUMMARY


,check,status,count,remarks
0,Duplicate malicious_id,PASS,0,
1,Missing Metadata,PASS,0,
2,Missing Labels,PASS,0,
3,Invalid Labels,PASS,0,
4,Empty Payload Text,PASS,0,
5,Invalid Attack Family,PASS,0,
6,Broken Image Paths,PASS,0,
7,One-to-One Mapping,PASS,0,
8,Orphan Images,PASS,0,
9,Split Group Leakage,PASS,0,


Total Checks : 10
Passed       : 10
Failed       : 0
[20:55:19] INFO : Dataset validation PASSED.


INFO:MODULE_D:Dataset validation PASSED.


[20:55:19] INFO : Validation report saved successfully.


INFO:MODULE_D:Validation report saved successfully.


In [ ]:
# ============================================================
# STEP 13 : FINAL PROJECT SUMMARY & COMPLETION
# ============================================================

print_stage("STEP 13 : FINAL PROJECT SUMMARY")

logger.info("Generating final project statistics...")

# ============================================================
# File Statistics
# ============================================================

parquet_size = (
    CONFIG["PARQUET"].stat().st_size
    if CONFIG["PARQUET"].exists()
    else 0
)

metadata_size = (
    CONFIG["METADATA"].stat().st_size
    if CONFIG["METADATA"].exists()
    else 0
)

validation_size = (
    CONFIG["VALIDATION"].stat().st_size
    if CONFIG["VALIDATION"].exists()
    else 0
)

manifest_size = (
    CONFIG["MANIFEST"].stat().st_size
    if CONFIG["MANIFEST"].exists()
    else 0
)

statistics_size = (
    CONFIG["STATISTICS"].stat().st_size
    if CONFIG["STATISTICS"].exists()
    else 0
)

# ============================================================
# Dataset Statistics
# ============================================================

total_samples = len(unified_metadata)

unique_benign = (
    unified_metadata["source_benign_id"].nunique()
    if "source_benign_id" in unified_metadata.columns
    else 0
)

unique_payloads = (
    unified_metadata["payload_id"].nunique()
    if "payload_id" in unified_metadata.columns
    else 0
)

unique_attack_families = (
    unified_metadata["attack_family"].nunique()
    if "attack_family" in unified_metadata.columns
    else 0
)

image_records = (
    unified_metadata["image_path"]
    .fillna("")
    .ne("")
    .sum()
    if "image_path" in unified_metadata.columns
    else 0
)

# ============================================================
# Attack Distribution
# ============================================================

print("\n")
print("=" * 90)
print("ATTACK FAMILY DISTRIBUTION")
print("=" * 90)

attack_distribution = (
    unified_metadata["attack_family"]
    .value_counts()
    .sort_values(ascending=False)
)

display(attack_distribution.to_frame("Samples"))

# ============================================================
# Label Distribution
# ============================================================

print("\n")
print("=" * 90)
print("LABEL DISTRIBUTION")
print("=" * 90)

label_distribution = (
    unified_metadata["label"]
    .value_counts()
)

display(label_distribution.to_frame("Samples"))

# ============================================================
# Split Distribution
# ============================================================

print("\n")
print("=" * 90)
print("SPLIT DISTRIBUTION")
print("=" * 90)

split_distribution = pd.DataFrame({
    "Split": ["train", "validation", "test"],
    "Samples": [len(train_df), len(val_df), len(test_df)],
    "Benign Groups": [len(train_ids), len(val_ids), len(test_ids)]
})

display(split_distribution)

# ============================================================
# Output Verification
# ============================================================

print("\n")
print("=" * 90)
print("OUTPUT VERIFICATION")
print("=" * 90)

outputs = {

    "Payload Bank":
        CONFIG["PAYLOAD_BANK"],

    "Malicious Text":
        CONFIG["MALICIOUS_TEXT"],

    "Metadata":
        CONFIG["METADATA"],

    "Parquet":
        CONFIG["PARQUET"],

    "Train Split":
        CONFIG["TRAIN_SPLIT"],

    "Validation Split":
        CONFIG["VAL_SPLIT"],

    "Test Split":
        CONFIG["TEST_SPLIT"],

    "Validation Report":
        CONFIG["VALIDATION"],

    "Statistics":
        CONFIG["STATISTICS"],

    "Manifest":
        CONFIG["MANIFEST"]

}

verification = []

for name, path in outputs.items():

    exists = path.exists()

    verification.append({

        "Output": name,

        "Exists": exists,

        "Path": str(path)

    })

verification_df = pd.DataFrame(verification)

display(verification_df)

# ============================================================
# Storage Summary
# ============================================================

print("\n")
print("=" * 90)
print("STORAGE SUMMARY")
print("=" * 90)

print(f"Metadata CSV        : {human_size(metadata_size)}")
print(f"Parquet Dataset     : {human_size(parquet_size)}")
print(f"Validation Report   : {human_size(validation_size)}")
print(f"Statistics          : {human_size(statistics_size)}")
print(f"Manifest            : {human_size(manifest_size)}")

print("=" * 90)
dataset_size = sum(

    f.stat().st_size

    for f in CONFIG["MASTER"].rglob("*")

    if f.is_file()

)

print(f"Complete Dataset     : {human_size(dataset_size)}")

# ============================================================
# Final Dataset Summary
# ============================================================

print("\n")
print("=" * 90)
print("MASTER_MALICIOUS_DATASET SUMMARY")
print("=" * 90)

print(f"Total Samples              : {total_samples:,}")
print(f"Unique Benign Documents    : {unique_benign:,}")
print(f"Unique Payloads            : {unique_payloads:,}")
print(f"Attack Families            : {unique_attack_families:,}")
print(f"Attack Types               : {unified_metadata['attack_name'].nunique():,}")
print(f"Image Samples              : {image_records:,}")
print(f"Modalities                 : Image + Text")

print("=" * 90)

# ============================================================
# Final Validation Status
# ============================================================

passed_checks = (
    validation_df["status"] == "PASS"
).sum()

failed_checks = (
    validation_df["status"] == "FAIL"
).sum()

print("\n")
print("=" * 90)
print("VALIDATION STATUS")
print("=" * 90)

print(f"Passed Checks : {passed_checks}")
print(f"Failed Checks : {failed_checks}")

print("=" * 90)

# ============================================================
# Completion
# ============================================================

logger.info("MASTER_MALICIOUS_DATASET successfully validated.")
logger.info("Dataset is ready for Agentic AI training pipeline.")
logger.info("Runtime OCR design preserved.")
logger.info("Storage-efficient pipeline verified.")
logger.info("Resume support verified.")

print("\n")
print("=" * 90)
print("🎉 MODULE D COMPLETED SUCCESSFULLY")
print("=" * 90)
print("MASTER_MALICIOUS_DATASET is ready for the next phase.")
print("Next Stage: Agentic AI Framework Training & Evaluation")
print("=" * 90)

print("MODULE D COMPLETED")



STEP 13 : FINAL PROJECT SUMMARY
[20:57:09] INFO : Generating final project statistics...


INFO:MODULE_D:Generating final project statistics...




ATTACK FAMILY DISTRIBUTION


,Samples
attack_family,
Context Poisoning,9663
Instruction Override,9663
Footer Injection,9663
Policy Bypass,9663
White Text Injection,9663
Metadata Injection,9663
Insurance-specific Attacks,9663
Hidden Multimodal,9662
Header Injection,9662




LABEL DISTRIBUTION


,Samples
label,
1,144937


... (11 outputs left out)

INFO:MODULE_D:Storage-efficient pipeline verified.


[20:57:11] INFO : Resume support verified.


INFO:MODULE_D:Resume support verified.




🎉 MODULE D COMPLETED SUCCESSFULLY
MASTER_MALICIOUS_DATASET is ready for the next phase.
Next Stage: Agentic AI Framework Training & Evaluation
MODULE D COMPLETED


In [ ]:
# ============================================================
# CREATE FINAL_DATASET (Freeze Version)
# ============================================================

from pathlib import Path
import shutil

MASTER = Path("/content/drive/MyDrive/MASTER_MALICIOUS_DATASET")
FINAL = Path("/content/drive/MyDrive/FINAL_DATASET")

print("="*80)
print("CREATING FINAL_DATASET")
print("="*80)

# ------------------------------------------------------------
# Fresh Folder
# ------------------------------------------------------------

if FINAL.exists():
    shutil.rmtree(FINAL)

FINAL.mkdir(parents=True)

# ------------------------------------------------------------
# Folder Structure
# ------------------------------------------------------------

(FINAL / "TEXT").mkdir()
(FINAL / "IMAGES").mkdir()
(FINAL / "LABELS").mkdir()
(FINAL / "PARQUET").mkdir()
(FINAL / "SPLITS").mkdir()
(FINAL / "PAYLOAD_BANK").mkdir()

# ------------------------------------------------------------
# TEXT
# ------------------------------------------------------------

shutil.copy2(
    MASTER / "TEXT" / "malicious_text.csv",
    FINAL / "TEXT" / "malicious_text.csv"
)

# ------------------------------------------------------------
# IMAGES
# ------------------------------------------------------------

if (MASTER / "IMAGE_SOURCES").exists():

    shutil.copytree(
        MASTER / "IMAGE_SOURCES",
        FINAL / "IMAGES",
        dirs_exist_ok=True
    )

# ------------------------------------------------------------
# LABELS
# ------------------------------------------------------------

shutil.copy2(
    MASTER / "LABELS" / "image_labels.csv",
    FINAL / "LABELS" / "image_labels.csv"
)

# ------------------------------------------------------------
# PARQUET
# ------------------------------------------------------------

shutil.copy2(
    MASTER / "PARQUET" / "malicious_dataset.parquet",
    FINAL / "PARQUET" / "malicious_dataset.parquet"
)

# ------------------------------------------------------------
# SPLITS
# ------------------------------------------------------------

for file in [

    "train.parquet",

    "validation.parquet",

    "test.parquet"

]:

    shutil.copy2(

        MASTER / "SPLITS" / file,

        FINAL / "SPLITS" / file

    )

# ------------------------------------------------------------
# PAYLOAD BANK
# ------------------------------------------------------------

shutil.copy2(

    MASTER / "PAYLOAD_BANK" / "insurance_payload_bank.csv",

    FINAL / "PAYLOAD_BANK" / "insurance_payload_bank.csv"

)

# ------------------------------------------------------------
# ROOT FILES
# ------------------------------------------------------------

for file in [

    "metadata.csv",

    "statistics.json",

    "dataset_manifest.json"

]:

    shutil.copy2(

        MASTER / file,

        FINAL / file

    )

# ------------------------------------------------------------
# VALIDATION REPORT
# ------------------------------------------------------------

shutil.copy2(

    MASTER / "VALIDATION" / "validation_report.csv",

    FINAL / "validation_report.csv"

)

# ============================================================
# VERIFY FINAL DATASET
# ============================================================

print("\n")
print("="*80)
print("FINAL_DATASET STRUCTURE")
print("="*80)

for path in sorted(FINAL.rglob("*")):

    print(path.relative_to(FINAL))

print("\n")

print("="*80)
print("FINAL_DATASET CREATED SUCCESSFULLY")
print("="*80)

print(f"Location : {FINAL}")

print("="*80)

CREATING FINAL_DATASET


FINAL_DATASET STRUCTURE
IMAGES
LABELS
LABELS/image_labels.csv
PARQUET
PARQUET/malicious_dataset.parquet
PAYLOAD_BANK
PAYLOAD_BANK/insurance_payload_bank.csv
SPLITS
SPLITS/test.parquet
SPLITS/train.parquet
SPLITS/validation.parquet
TEXT
TEXT/malicious_text.csv
dataset_manifest.json
metadata.csv
statistics.json
validation_report.csv


FINAL_DATASET CREATED SUCCESSFULLY
Location : /content/drive/MyDrive/FINAL_DATASET
